# AdireTrust — staged training on Google Colab

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/drizzy765/adiretrust-app/blob/main/model/train_colab.ipynb)

Train the cloth-check classifier (hand-dyed Adire vs factory print) on a Colab GPU, then export a small probe for the FastAPI app. This notebook follows the official pretraining / engineering plan.

**Sources of truth**

- Notion [AdireTrust — Hackathon Prep Plan](https://app.notion.com/p/4199e1028e8581b4a2a6f210c2a32cf8) §8 (Staged Transfer Learning) and §3 (experiments)
- [`docs/research-plan.md`](https://github.com/drizzy765/adiretrust-app/blob/main/docs/research-plan.md) §6–7
- [`docs/model-methodology.md`](https://github.com/drizzy765/adiretrust-app/blob/main/docs/model-methodology.md)
- [`docs/PLAN.md`](https://github.com/drizzy765/adiretrust-app/blob/main/docs/PLAN.md)

The same plan is in the research repo’s `adiretrust.md` / `adiretrust.pdf`. The [ajibua/AdireTrust README](https://github.com/ajibua/AdireTrust) restates the four-stage table. There is **no trained AdireTrust accuracy result** in this repository. Do not invent one.

---

**Data stages**

1. **General backbone** — frozen pretrained DINOv2 or SigLIP (ViT-S / ViT-B). Not trained from scratch.
2. **Intermediate related task** — batik tulis/cap vs print, plus DTD, AITEX, optionally Wax-MNIST / TILDA. These are **not** auto-downloaded here.
3. **Scraped + synthetic** — labeled web images and procedural textures. **Synthetic is train only; evaluate on real photos only.**
4. **Field photos** — Abeokuta / Osun genuine vs print. This is the demo domain and the **default run**.

**Experiment order (plan)**

1. Zero-shot CLIP / SigLIP with the plan prompts (optional cell, off by default).
2. Linear probe **or** k-NN on frozen features. Try DINOv2, SigLIP, and a frozen EfficientNet. Try global embeddings and patch tokens. Compare on **val**. This is the first real score.
3. Fine-tune **only the val winner**, last blocks first (~1e-5, early stopping), and only if the probe is useful and *n* is large enough. Do not full-network fine-tune on fewer than 300 pieces.
4. Ablations if time: front vs front+back; full vs macro; one phone vs mixed.

**Default run (this notebook, cells 1–11):** frozen **DINOv2 ViT-S/14** + **logistic regression**. Keep that path even if you later turn on optional cells.

**Optional cells (off by default).** Set the matching `RUN_*` flag in the flags cell, then run that section:

- k-NN vs logistic regression
- SigLIP and frozen EfficientNet-B0 probes
- DINOv2 ViT-B and patch tokens
- CLIP / SigLIP zero-shot with the plan prompts
- Stage-2 DTD / batik note (no auto-download)
- Train-only augmentation + last-block fine-tune (**OFF**; warning about small *n*)
- DINOv2 patch-similarity heatmap
- Multi-shot aggregation when `view_tag` is real (skipped while tags are `unknown`)

**Data.** Image files are **not** in git. Section 3 downloads the public Drive folder **AdireTrust-data** with `gdown --folder` (default), or you can mount Drive. Folder: https://drive.google.com/drive/folders/1k1j2mNU1H5ST74AJp3qGEly67kEdnPNN — layout `raw/osun_genuine`, `raw/osun_prints`, `raw/synthetic`. The 100 print photos may still be missing from `raw/osun_prints/`. Do not invent print images or metrics.

Set the runtime to **GPU** (Runtime → Change runtime type) before running.


## 1. Check the GPU


In [1]:
import sys

print("Python", sys.version)

try:
    import torch
    print("torch", torch.__version__)
    cuda = torch.cuda.is_available()
    print("CUDA available:", cuda)
    if cuda:
        print("GPU:", torch.cuda.get_device_name(0))
    else:
        print("No GPU. Runtime → Change runtime type → GPU, then rerun this cell.")
except ImportError:
    print("torch is not installed yet. The next cell will install CPU/GPU wheels as needed.")


Python 3.13.16 (main, Oct  1 2026, 15:40:24) [GCC 13.3.0]
torch 2.14.1+cu130
CUDA available: True
GPU: Tesla T4


## 2. Clone the repo and install training packages

If you already opened this notebook from the cloned repo, the cell reuses that copy. Otherwise it clones `drizzy765/adiretrust-app`.


In [2]:
from pathlib import Path
import os
import subprocess
import sys

REPO_URL = "https://github.com/drizzy765/adiretrust-app.git"
REPO_NAME = "adiretrust-app"


def already_in_repo() -> Path | None:
    here = Path.cwd().resolve()
    for candidate in [here, *here.parents]:
        if (candidate / "data" / "labels" / "labels.csv").is_file() and (
            candidate / "model" / "train_colab.ipynb"
        ).is_file():
            return candidate
    return None


repo = already_in_repo()
if repo is None:
    dest = Path.cwd() / REPO_NAME
    if not dest.exists():
        subprocess.check_call(["git", "clone", "--depth", "1", REPO_URL, str(dest)])
    repo = dest
    os.chdir(repo)
else:
    os.chdir(repo)

REPO_ROOT = Path.cwd().resolve()
print("REPO_ROOT =", REPO_ROOT)

%pip install -q scikit-learn pandas pillow joblib timm numpy matplotlib

try:
    import torch  # noqa: F401
except ImportError:
    %pip install -q torch torchvision --index-url https://download.pytorch.org/whl/cu121


REPO_ROOT = /content/adiretrust-app


## 3. Get the training images (default: per-file download from the manifest)

Images are **not** in the git repo. They live in the public Drive folder **AdireTrust-data** (Anyone with the link, Viewer):

https://drive.google.com/drive/folders/1k1j2mNU1H5ST74AJp3qGEly67kEdnPNN

Expected layout:

```
AdireTrust-data/
  raw/osun_genuine/
  raw/osun_prints/
  raw/synthetic/
```

**Default:** read `data/drive_manifest.csv` (one Drive file id per labeled image) and download each file into `/content/AdireTrust-data/raw/<subfolder>/`. This avoids `gdown --folder`'s 50-file-per-folder cap so all 205 labeled images are loaded. Existing files are skipped.

**Alternative:** set `USE_DRIVE_MOUNT = True` if `AdireTrust-data` is already in your My Drive (or as a shortcut) at `/content/drive/MyDrive/AdireTrust-data`.

**Last fallback:** set `USE_GDOWN_FOLDER = True` to use `gdown --folder` (capped at 50 files per folder).

Repo paths with the same layout are used if Drive is unavailable. This cell never invents images.


In [3]:
from pathlib import Path
import csv
import shutil
import subprocess
import sys
from collections import Counter
from concurrent.futures import ThreadPoolExecutor, as_completed

DRIVE_FOLDER_URL = "https://drive.google.com/drive/folders/1k1j2mNU1H5ST74AJp3qGEly67kEdnPNN"
DRIVE_FOLDER_NAME = "AdireTrust-data"

# Default: per-file download from data/drive_manifest.csv (avoids the 50-file folder cap).
# Alternative: mount My Drive. Last fallback: gdown --folder.
USE_DRIVE_MOUNT = False
USE_GDOWN_FOLDER = False
DOWNLOAD_WORKERS = 12

LOCAL_DATA_ROOT = Path("/content/AdireTrust-data")
MOUNTED_DATA_ROOT = Path("/content/drive/MyDrive/AdireTrust-data")
MANIFEST_PATH = REPO_ROOT / "data" / "drive_manifest.csv"

# Optional overrides if a class lives in a different folder. Leave blank to use DATA_DRIVE_ROOT.
PRINTS_DRIVE_DIR = ""
GENUINE_DRIVE_DIR = ""
SYNTHETIC_DRIVE_DIR = ""

PRINTS_LOCAL_DIR = REPO_ROOT / "data" / "raw" / "osun_prints"
GENUINE_LOCAL_DIR = REPO_ROOT / "data" / "raw" / "osun_genuine"
SYNTHETIC_LOCAL_DIR = REPO_ROOT / "data" / "raw" / "synthetic"


def _count_images(folder: Path) -> int:
    if not folder.is_dir():
        return 0
    return sum(
        1
        for p in folder.iterdir()
        if p.is_file() and p.suffix.lower() in {".jpg", ".jpeg", ".png", ".webp"}
    )


def _has_layout(root: Path) -> bool:
    return (root / "raw" / "osun_genuine").is_dir() or (
        root / "data" / "raw" / "osun_genuine"
    ).is_dir()


def _normalize_root(root: Path) -> Path:
    # gdown may write either root/raw/... or root/AdireTrust-data/raw/...
    nested = root / DRIVE_FOLDER_NAME
    if not _has_layout(root) and _has_layout(nested):
        return nested
    return root


def _resolve_class_dir(root: Path, name: str) -> Path | None:
    for cand in (root / "raw" / name, root / "data" / "raw" / name, root / name):
        if cand.is_dir():
            return cand
    return None


def _maybe_mount_drive() -> None:
    try:
        from google.colab import drive  # type: ignore

        drive.mount("/content/drive")
    except ImportError:
        print("Not running in Colab; using filesystem paths as-is.")


def _sync_image_dir(src: Path, dest: Path) -> int:
    if not src.exists():
        print(f"Source does not exist, skip sync: {src}")
        return 0
    dest.mkdir(parents=True, exist_ok=True)
    copied = 0
    for p in src.rglob("*"):
        if not p.is_file() or p.suffix.lower() not in {".jpg", ".jpeg", ".png", ".webp"}:
            continue
        target = dest / p.name
        if target.exists() or target.is_symlink():
            continue
        try:
            target.symlink_to(p.resolve())
        except OSError:
            shutil.copy2(p, target)
        copied += 1
    return copied


def _ensure_gdown() -> None:
    try:
        import gdown  # noqa: F401
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "gdown"])


def _download_one(file_id: str, dest: Path) -> str:
    """Download one public Drive file by id. Returns 'skip', 'ok', or 'fail'."""
    dest.parent.mkdir(parents=True, exist_ok=True)
    if dest.exists() and dest.stat().st_size > 0:
        return "skip"
    import gdown

    tmp = dest.with_name(dest.name + ".part")
    if tmp.exists():
        tmp.unlink()
    try:
        out = gdown.download(id=file_id, output=str(tmp), quiet=True)
        src = Path(out) if out else tmp
        if src.exists() and src.stat().st_size > 0:
            src.replace(dest)
            return "ok"
    except Exception:
        if tmp.exists():
            tmp.unlink()
    return "fail"


def _download_from_manifest(manifest_path: Path, dest_root: Path, workers: int = 12) -> dict[str, Counter]:
    if not manifest_path.is_file():
        raise FileNotFoundError(f"Missing manifest: {manifest_path}")
    rows = list(csv.DictReader(manifest_path.open(encoding="utf-8")))
    required = {"subfolder", "filename", "file_id"}
    if not rows or required - set(rows[0].keys()):
        raise ValueError(f"Manifest must have columns {sorted(required)}: {manifest_path}")
    print(f"Downloading {len(rows)} files from {manifest_path} -> {dest_root / 'raw'} ({workers} workers)")

    def job(row: dict) -> tuple[str, str]:
        sub = row["subfolder"].strip()
        name = row["filename"].strip()
        file_id = row["file_id"].strip()
        dest = dest_root / "raw" / sub / name
        status = _download_one(file_id, dest)
        if status == "fail":
            status = _download_one(file_id, dest)  # retry once
            if status == "ok":
                status = "ok_retry"
        return sub, status

    stats: dict[str, Counter] = {}
    with ThreadPoolExecutor(max_workers=workers) as pool:
        futures = [pool.submit(job, row) for row in rows]
        for fut in as_completed(futures):
            sub, status = fut.result()
            stats.setdefault(sub, Counter())[status] += 1

    for sub in ("osun_genuine", "osun_prints", "synthetic"):
        c = stats.get(sub, Counter())
        downloaded = c.get("ok", 0) + c.get("ok_retry", 0)
        skipped = c.get("skip", 0)
        failed = c.get("fail", 0)
        print(
            f"{sub}: downloaded={downloaded} skipped={skipped} failed={failed} "
            f"present={_count_images(dest_root / 'raw' / sub)}"
        )
    failed_total = sum(c.get("fail", 0) for c in stats.values())
    if failed_total:
        print(
            f"WARNING: {failed_total} file(s) failed after one retry. "
            "Set USE_DRIVE_MOUNT = True if you own the folder, or rerun this cell."
        )
    return stats


DATA_DRIVE_ROOT = ""

if USE_DRIVE_MOUNT:
    _maybe_mount_drive()
    DATA_DRIVE_ROOT = str(MOUNTED_DATA_ROOT)
    print("Using mounted Drive at", DATA_DRIVE_ROOT)
    if not Path(DATA_DRIVE_ROOT).is_dir():
        print(
            "Mounted path is missing. In Drive, add a shortcut to AdireTrust-data "
            "under My Drive, or leave USE_DRIVE_MOUNT = False to download by file id."
        )
elif USE_GDOWN_FOLDER:
    _ensure_gdown()
    print("Fallback: downloading public folder with gdown --folder (50-file cap per folder):")
    print(f"  gdown --folder {DRIVE_FOLDER_URL} -O {LOCAL_DATA_ROOT}")
    LOCAL_DATA_ROOT.mkdir(parents=True, exist_ok=True)
    if _has_layout(_normalize_root(LOCAL_DATA_ROOT)):
        print("Images already present under", _normalize_root(LOCAL_DATA_ROOT))
    else:
        !gdown --folder https://drive.google.com/drive/folders/1k1j2mNU1H5ST74AJp3qGEly67kEdnPNN -O /content/AdireTrust-data --remaining-ok
    root = _normalize_root(LOCAL_DATA_ROOT)
    if not _has_layout(root):
        print(
            "gdown --folder did not produce the expected layout. "
            "Confirm the Drive folder is Anyone with the link (Viewer), "
            "or leave USE_GDOWN_FOLDER = False to download by file id."
        )
    DATA_DRIVE_ROOT = str(root)
    print("DATA_DRIVE_ROOT =", DATA_DRIVE_ROOT)
else:
    _ensure_gdown()
    LOCAL_DATA_ROOT.mkdir(parents=True, exist_ok=True)
    if not MANIFEST_PATH.is_file():
        print(
            f"Manifest not found at {MANIFEST_PATH}. "
            "Falling back to gdown --folder (50-file cap per folder)."
        )
        if not _has_layout(_normalize_root(LOCAL_DATA_ROOT)):
            !gdown --folder https://drive.google.com/drive/folders/1k1j2mNU1H5ST74AJp3qGEly67kEdnPNN -O /content/AdireTrust-data --remaining-ok
    else:
        _download_from_manifest(MANIFEST_PATH, LOCAL_DATA_ROOT, workers=DOWNLOAD_WORKERS)
    root = _normalize_root(LOCAL_DATA_ROOT)
    DATA_DRIVE_ROOT = str(root)
    print("DATA_DRIVE_ROOT =", DATA_DRIVE_ROOT)

drive_root = Path(DATA_DRIVE_ROOT).expanduser() if DATA_DRIVE_ROOT else None
if drive_root and drive_root.is_dir():
    for folder, dest in [
        ("osun_prints", PRINTS_LOCAL_DIR),
        ("osun_genuine", GENUINE_LOCAL_DIR),
        ("synthetic", SYNTHETIC_LOCAL_DIR),
    ]:
        src = _resolve_class_dir(drive_root, folder)
        if src is None:
            print(f"DATA_DRIVE_ROOT: no folder for {folder} under {drive_root}")
            continue
        n = _sync_image_dir(src, dest)
        print(f"DATA_DRIVE_ROOT: linked/copied {n} new files from {src} -> {dest}")

for name, folder in [
    ("osun_genuine", GENUINE_LOCAL_DIR),
    ("osun_prints", PRINTS_LOCAL_DIR),
    ("synthetic", SYNTHETIC_LOCAL_DIR),
]:
    print(f"{name}: exists={folder.is_dir()} n_images={_count_images(folder)} path={folder}")


osun_genuine: downloaded=81 skipped=0 failed=0 present=81
osun_prints: downloaded=99 skipped=0 failed=1 present=99
synthetic: downloaded=24 skipped=0 failed=0 present=24
DATA_DRIVE_ROOT = /content/AdireTrust-data
DATA_DRIVE_ROOT: linked/copied 99 new files from /content/AdireTrust-data/raw/osun_prints -> /content/adiretrust-app/data/raw/osun_prints
DATA_DRIVE_ROOT: linked/copied 81 new files from /content/AdireTrust-data/raw/osun_genuine -> /content/adiretrust-app/data/raw/osun_genuine
DATA_DRIVE_ROOT: linked/copied 24 new files from /content/AdireTrust-data/raw/synthetic -> /content/adiretrust-app/data/raw/synthetic
osun_genuine: exists=True n_images=81 path=/content/adiretrust-app/data/raw/osun_genuine
osun_prints: exists=True n_images=99 path=/content/adiretrust-app/data/raw/osun_prints
synthetic: exists=True n_images=24 path=/content/adiretrust-app/data/raw/synthetic


## 4. Load labels, splits, Dataset / loaders, and experiment flags

Reads `data/labels/labels.csv` and `splits.csv` from the clone. Resolves each image from **DATA_DRIVE_ROOT first** (`raw/osun_genuine`, `raw/osun_prints`, `raw/synthetic`, or `data/raw/...` under the gdown / mounted Drive folder), then falls back to the same paths in this repo (plus processed 224/512 if present). Builds a `torch.utils.data.Dataset` and train/val/test loaders, and prints counts per class/split plus missing files.

Synthetics stay in **train** only. Unsplit Commons rows are skipped. Every `RUN_*` flag defaults to `False`; the default DINOv2 ViT-S/14 + logistic regression path always runs.


In [4]:
from __future__ import annotations

from collections import Counter
from pathlib import Path
import random

import numpy as np
import pandas as pd
from PIL import Image, ImageEnhance, ImageFilter
import torch
from torch.utils.data import DataLoader, Dataset

LABELS_PATH = REPO_ROOT / "data" / "labels" / "labels.csv"
SPLITS_PATH = REPO_ROOT / "data" / "labels" / "splits.csv"
WEIGHTS_DIR = REPO_ROOT / "model" / "weights"
RUNS_DIR = REPO_ROOT / "model" / "runs"
WEIGHTS_DIR.mkdir(parents=True, exist_ok=True)
RUNS_DIR.mkdir(parents=True, exist_ok=True)

CLASS_ORDER = ["factory_print", "genuine_hand_dyed"]
VALID_LABELS = tuple(CLASS_ORDER)

# Optional experiments: set a flag to True, then run the matching later cell.
RUN_KNN = False
RUN_SIGLIP_EFFICIENTNET = False
RUN_DINOV2_VITB_PATCH = False
RUN_ZEROSHOT = False
RUN_STAGE2 = False  # never auto-downloads batik / DTD / AITEX
RUN_FINETUNE = False  # keep OFF on this small set unless you have a reason
RUN_HEATMAP = False
RUN_MULTISHOT = False  # auto-skips when every view_tag is unknown

CONFIDENCE_THRESHOLD_DEFAULT = 0.6  # app starting point; also swept on val
THRESHOLD_GRID = [0.50, 0.55, 0.60, 0.65, 0.70, 0.75, 0.80]
IMAGE_SIZE = 224
CACHE_NAME = "dinov2_vits14_224"
BATCH_SIZE = 16
IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)

labels = pd.read_csv(LABELS_PATH)
splits = pd.read_csv(SPLITS_PATH)
manifest = labels.merge(splits, on="piece_id", how="left")

print("labels:", len(labels), "split rows:", len(splits), "merged:", len(manifest))
print("label counts:\n", labels["label"].value_counts(dropna=False).to_string())
print("label_source counts:\n", labels["label_source"].value_counts(dropna=False).to_string())
print("split counts (including unsplit):\n", manifest["split"].value_counts(dropna=False).to_string())


def _image_suffix(path: Path) -> bool:
    return path.is_file() and path.suffix.lower() in {".jpg", ".jpeg", ".png", ".webp"}


def candidate_paths(row: dict) -> list[Path]:
    rel = str(row.get("filename") or "").replace("\\", "/").lstrip("/")
    name = Path(rel).name
    drive_prints = Path(str(globals().get("PRINTS_DRIVE_DIR") or "")).expanduser() if globals().get("PRINTS_DRIVE_DIR") else None
    drive_genuine = Path(str(globals().get("GENUINE_DRIVE_DIR") or "")).expanduser() if globals().get("GENUINE_DRIVE_DIR") else None
    drive_synth = Path(str(globals().get("SYNTHETIC_DRIVE_DIR") or "")).expanduser() if globals().get("SYNTHETIC_DRIVE_DIR") else None
    drive_root = Path(str(globals().get("DATA_DRIVE_ROOT") or "")).expanduser() if globals().get("DATA_DRIVE_ROOT") else None

    cands: list[Path] = []
    # Primary: DATA_DRIVE_ROOT (gdown download or mounted AdireTrust-data)
    if drive_root:
        cands.extend(
            [
                drive_root / "raw" / "osun_genuine" / name,
                drive_root / "raw" / "osun_prints" / name,
                drive_root / "raw" / "synthetic" / name,
                drive_root / rel,
                drive_root / "data" / "raw" / "osun_genuine" / name,
                drive_root / "data" / "raw" / "osun_prints" / name,
                drive_root / "data" / "raw" / "synthetic" / name,
                drive_root / "osun_genuine" / name,
                drive_root / "osun_prints" / name,
                drive_root / "synthetic" / name,
            ]
        )
    if drive_prints:
        cands.append(drive_prints / name)
    if drive_genuine:
        cands.append(drive_genuine / name)
    if drive_synth:
        cands.append(drive_synth / name)
    # Fallback: files already in the clone or synced into the repo tree
    cands.extend(
        [
            REPO_ROOT / rel,
            REPO_ROOT / "data" / rel,
            REPO_ROOT / "data" / "raw" / "osun_genuine" / name,
            REPO_ROOT / "data" / "raw" / "osun_prints" / name,
            REPO_ROOT / "data" / "raw" / "synthetic" / name,
            REPO_ROOT / "data" / "processed" / "osun_workshop" / "224" / name,
            REPO_ROOT / "data" / "processed" / "osun_workshop" / "512" / name,
        ]
    )
    return cands


def resolve_image(row: dict) -> Path | None:
    seen: set[str] = set()
    for path in candidate_paths(row):
        key = str(path)
        if key in seen:
            continue
        seen.add(key)
        if _image_suffix(path):
            return path
    return None


resolved: list[dict] = []
missing: list[dict] = []
usable: list[dict] = []
skipped = Counter()

for row in manifest.to_dict(orient="records"):
    rec = {
        "piece_id": str(row["piece_id"]),
        "image_id": str(row["piece_id"]),
        "label": row.get("label"),
        "source": str(row.get("label_source") or ""),
        "view_tag": str(row.get("view_tag") or ""),
        "filename": str(row.get("filename") or "").replace("\\", "/"),
        "split": row.get("split"),
    }
    path = resolve_image(row)
    rec["resolved_path"] = str(path) if path else ""
    rec["path"] = rec["resolved_path"]
    rec["found"] = path is not None

    source = rec["source"]
    split = rec["split"]
    if source == "wikimedia_commons":
        skipped["commons"] += 1
        if not rec["found"]:
            skipped["commons_missing_file"] += 1
        continue
    if pd.isna(split) or str(split).strip() == "":
        skipped["unsplit"] += 1
        if rec["found"]:
            resolved.append(rec)
        else:
            missing.append(rec)
        continue
    rec["split"] = str(split)
    if source == "synthetic" and rec["split"] != "train":
        skipped["synthetic_not_train"] += 1
        continue
    if rec["label"] not in VALID_LABELS:
        skipped["uncertain_or_other"] += 1
        continue
    if not rec["found"]:
        skipped["missing_file"] += 1
        missing.append(rec)
        continue
    resolved.append(rec)
    usable.append(rec)

print("resolved usable image files:", len(usable))
print("missing (non-commons, valid label, has split):", len(missing), "skipped:", dict(skipped))
print("missing by source:", dict(Counter(r["source"] for r in missing)))
print("missing by label:", dict(Counter(str(r["label"]) for r in missing)))
if missing:
    print("first missing filenames:")
    for rec in missing[:10]:
        print(" ", rec["label"], rec["source"], rec["filename"])

print("usable by split:", dict(Counter(r["split"] for r in usable)))
print("usable by label:", dict(Counter(r["label"] for r in usable)))
print("usable by source:", dict(Counter(r["source"] for r in usable)))
print("train sources:", dict(Counter(r["source"] for r in usable if r["split"] == "train")))

split_labels = {
    s: dict(Counter(r["label"] for r in usable if r["split"] == s))
    for s in ("train", "val", "test")
}
print("label counts by split:", split_labels)

n_classes_train = len(split_labels["train"])
BOTH_CLASSES = n_classes_train >= 2
if skipped.get("missing_file"):
    print(
        f"\nWARNING: {skipped['missing_file']} manifest rows have no image file. "
        "Add missing photos to the AdireTrust-data Drive folder "
        "(raw/osun_prints, raw/osun_genuine, raw/synthetic) and rerun section 3. "
        "Do not invent a print set or an accuracy number."
    )
if not BOTH_CLASSES:
    print(
        "\nWARNING: training images cover only "
        f"{split_labels['train'] or 'no labels'}. "
        "A two-class probe cannot be trained until print photos are added. "
        "Later default cells will extract features if possible and skip fitting."
    )


def load_rgb_resized(path: str | Path, size: int = IMAGE_SIZE) -> Image.Image:
    with Image.open(path) as im:
        return im.convert("RGB").resize((size, size), Image.Resampling.BICUBIC)


def apply_train_augment(im: Image.Image) -> Image.Image:
    # Light train-only augmentation. Does not invent dye bleed or reverse-side cues.
    if random.random() < 0.5:
        angle = random.uniform(-12, 12)
        im = im.rotate(angle, resample=Image.Resampling.BICUBIC, fillcolor=(0, 0, 0))
    if random.random() < 0.3:
        im = im.filter(ImageFilter.GaussianBlur(radius=random.uniform(0.2, 1.1)))
    if random.random() < 0.7:
        im = ImageEnhance.Brightness(im).enhance(random.uniform(0.85, 1.15))
        im = ImageEnhance.Contrast(im).enhance(random.uniform(0.85, 1.15))
        im = ImageEnhance.Color(im).enhance(random.uniform(0.9, 1.1))
    if random.random() < 0.3:
        # Mild random crop, then back to size. Not a "macro invented" dye cue.
        w, h = im.size
        scale = random.uniform(0.85, 1.0)
        nw, nh = int(w * scale), int(h * scale)
        left = random.randint(0, w - nw) if nw < w else 0
        top = random.randint(0, h - nh) if nh < h else 0
        im = im.crop((left, top, left + nw, top + nh)).resize((w, h), Image.Resampling.BICUBIC)
    return im


def pil_to_normalized_tensor(im: Image.Image, mean=IMAGENET_MEAN, std=IMAGENET_STD) -> torch.Tensor:
    arr = np.asarray(im, dtype=np.float32) / 255.0
    x = torch.from_numpy(arr).permute(2, 0, 1)
    mean_t = torch.tensor(mean, dtype=x.dtype)[:, None, None]
    std_t = torch.tensor(std, dtype=x.dtype)[:, None, None]
    return (x - mean_t) / std_t


class AdireClothDataset(Dataset):
    # Train/val/test cloth photos from the resolved manifest.

    def __init__(
        self,
        records: list[dict],
        split: str | None = None,
        size: int = IMAGE_SIZE,
        mean=IMAGENET_MEAN,
        std=IMAGENET_STD,
        augment: bool = False,
        class_names: list[str] | None = None,
    ) -> None:
        self.records = [r for r in records if split is None or r["split"] == split]
        self.split = split
        self.size = size
        self.mean = mean
        self.std = std
        self.augment = augment
        self.class_names = list(class_names or CLASS_ORDER)

    def __len__(self) -> int:
        return len(self.records)

    def __getitem__(self, idx: int):
        rec = self.records[idx]
        im = load_rgb_resized(rec["resolved_path"], size=self.size)
        if self.augment:
            im = apply_train_augment(im)
        x = pil_to_normalized_tensor(im, mean=self.mean, std=self.std)
        if rec["label"] not in self.class_names:
            raise KeyError(f"Unexpected label {rec['label']!r} for {rec['piece_id']}")
        y = self.class_names.index(rec["label"])
        return x, y, rec["piece_id"]


def adire_collate(batch):
    xs, ys, ids = zip(*batch)
    return torch.stack(xs, dim=0), torch.tensor(ys, dtype=torch.long), list(ids)


def report_dataset(ds: AdireClothDataset) -> dict:
    counts = Counter(r["label"] for r in ds.records)
    sources = Counter(r["source"] for r in ds.records)
    print(f"  {ds.split or 'all'}: n={len(ds)}  labels={dict(counts)}  sources={dict(sources)}")
    return {"n": len(ds), "labels": dict(counts), "sources": dict(sources)}


def build_split_datasets(records: list[dict], augment_train: bool = False, batch_size: int = BATCH_SIZE):
    train_ds = AdireClothDataset(records, "train", augment=augment_train)
    val_ds = AdireClothDataset(records, "val", augment=False)
    test_ds = AdireClothDataset(records, "test", augment=False)
    print("=== Dataset / loader counts ===")
    stats = {name: report_dataset(ds) for name, ds in [("train", train_ds), ("val", val_ds), ("test", test_ds)]}
    print("missing files (not in any loader):", len(missing), dict(Counter(r["source"] for r in missing)))
    loaders = {
        "train": DataLoader(
            train_ds,
            batch_size=batch_size,
            shuffle=True,
            num_workers=0,
            collate_fn=adire_collate,
        ),
        "val": DataLoader(
            val_ds,
            batch_size=batch_size,
            shuffle=False,
            num_workers=0,
            collate_fn=adire_collate,
        ),
        "test": DataLoader(
            test_ds,
            batch_size=batch_size,
            shuffle=False,
            num_workers=0,
            collate_fn=adire_collate,
        ),
    }
    # One batch smoke test when train is non-empty (proves the loader is wired).
    if len(train_ds):
        xb, yb, ids = next(iter(loaders["train"]))
        print(f"train loader smoke: batch x={tuple(xb.shape)} y={tuple(yb.shape)} n_ids={len(ids)}")
    return train_ds, val_ds, test_ds, loaders, stats


train_ds, val_ds, test_ds, loaders, dataset_stats = build_split_datasets(usable, augment_train=False)
train_loader, val_loader, test_loader = loaders["train"], loaders["val"], loaders["test"]

print("\nFlags:")
for name, value in [
    ("RUN_KNN", RUN_KNN),
    ("RUN_SIGLIP_EFFICIENTNET", RUN_SIGLIP_EFFICIENTNET),
    ("RUN_DINOV2_VITB_PATCH", RUN_DINOV2_VITB_PATCH),
    ("RUN_ZEROSHOT", RUN_ZEROSHOT),
    ("RUN_STAGE2", RUN_STAGE2),
    ("RUN_FINETUNE", RUN_FINETUNE),
    ("RUN_HEATMAP", RUN_HEATMAP),
    ("RUN_MULTISHOT", RUN_MULTISHOT),
]:
    print(f"  {name} = {value}")


labels: 221 split rows: 205 merged: 221
label counts:
 label
factory_print        116
genuine_hand_dyed     99
uncertain              6
label_source counts:
 label_source
osun_artisan_workshop    181
synthetic                 24
wikimedia_commons         16
split counts (including unsplit):
 split
train    151
test      27
val       27
NaN       16
resolved usable image files: 204
missing (non-commons, valid label, has split): 1 skipped: {'commons': 16, 'commons_missing_file': 16, 'missing_file': 1}
missing by source: {'osun_artisan_workshop': 1}
missing by label: {'factory_print': 1}
first missing filenames:
  factory_print osun_artisan_workshop data/raw/osun_prints/IMG-20261123-WA0087.jpg
usable by split: {'train': 151, 'test': 27, 'val': 26}
usable by label: {'genuine_hand_dyed': 89, 'factory_print': 115}
usable by source: {'osun_artisan_workshop': 180, 'synthetic': 24}
train sources: {'osun_artisan_workshop': 127, 'synthetic': 24}
label counts by split: {'train': {'genuine_hand_dye

## 5. Default: extract frozen DINOv2 ViT-S/14 features

Images are resized to 224×224 and normalized with ImageNet mean/std (the setting DINOv2 expects). Features are cached under `model/runs/` so reruns do not re-extract. The backbone stays **frozen**.


In [5]:
from pathlib import Path
import numpy as np
from PIL import Image
import torch
import torch.nn.functional as F

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("feature device:", DEVICE)


def load_rgb_224(path: str | Path, size: int = IMAGE_SIZE) -> Image.Image:
    with Image.open(path) as im:
        return im.convert("RGB").resize((size, size), Image.Resampling.BICUBIC)


def pil_to_normalized_tensor(im: Image.Image, mean=IMAGENET_MEAN, std=IMAGENET_STD) -> torch.Tensor:
    arr = np.asarray(im, dtype=np.float32) / 255.0
    x = torch.from_numpy(arr).permute(2, 0, 1)
    mean_t = torch.tensor(mean, dtype=x.dtype)[:, None, None]
    std_t = torch.tensor(std, dtype=x.dtype)[:, None, None]
    return (x - mean_t) / std_t


def load_dinov2_vits14():
    errors = []
    try:
        model = torch.hub.load("facebookresearch/dinov2", "dinov2_vits14", pretrained=True)
        model.eval().to(DEVICE)
        for p in model.parameters():
            p.requires_grad = False
        print("Loaded facebookresearch/dinov2 dinov2_vits14")
        return model, "hub"
    except Exception as exc:  # noqa: BLE001
        errors.append(f"torch.hub: {exc}")
    try:
        import timm

        model = timm.create_model(
            "vit_small_patch14_dinov2.lvd142m",
            pretrained=True,
            num_classes=0,
        )
        model.eval().to(DEVICE)
        for p in model.parameters():
            p.requires_grad = False
        print("Loaded timm vit_small_patch14_dinov2.lvd142m")
        return model, "timm"
    except Exception as exc:  # noqa: BLE001
        errors.append(f"timm: {exc}")
    raise RuntimeError("Could not load DINOv2 ViT-S/14:\n" + "\n".join(errors))


@torch.no_grad()
def embed_batch_dinov2(model, kind: str, batch: torch.Tensor) -> np.ndarray:
    if kind == "hub":
        out = model(batch)
        if isinstance(out, dict):
            out = out.get("x_norm_clstoken", next(iter(out.values())))
        return out.detach().cpu().numpy()
    out = model(batch)
    if isinstance(out, (tuple, list)):
        out = out[0]
    return out.detach().cpu().numpy()


def extract_features(model, kind: str, records: list[dict], cache_stem: str, batch_size: int = 16) -> dict:
    cache_path = RUNS_DIR / f"{cache_stem}.npz"
    key_list = [(r["image_id"], r["split"], r["resolved_path"]) for r in records]
    if cache_path.is_file():
        cached = np.load(cache_path, allow_pickle=True)
        cached_keys = [tuple(x) for x in cached["keys"].tolist()]
        if cached_keys == key_list:
            print(f"Loaded cached features from {cache_path}  shape={cached['features'].shape}")
            return {
                "features": cached["features"],
                "y": cached["y"],
                "ids": cached["ids"],
                "splits": cached["splits"],
                "paths": cached["paths"],
            }
        print("Cache key mismatch; re-extracting.")

    feats = []
    batch_tensors = []
    for rec in records:
        im = load_rgb_224(rec["resolved_path"])
        batch_tensors.append(pil_to_normalized_tensor(im))
        if len(batch_tensors) == batch_size:
            x = torch.stack(batch_tensors, dim=0).to(DEVICE)
            feats.append(embed_batch_dinov2(model, kind, x))
            batch_tensors = []
    if batch_tensors:
        x = torch.stack(batch_tensors, dim=0).to(DEVICE)
        feats.append(embed_batch_dinov2(model, kind, x))

    features = np.concatenate(feats, axis=0) if feats else np.zeros((0, 1), dtype=np.float32)
    y = np.array([r["label"] for r in records], dtype=object)
    ids = np.array([r["image_id"] for r in records], dtype=object)
    split_arr = np.array([r["split"] for r in records], dtype=object)
    paths = np.array([r["resolved_path"] for r in records], dtype=object)
    np.savez_compressed(
        cache_path,
        features=features,
        y=y,
        ids=ids,
        splits=split_arr,
        paths=paths,
        keys=np.array(key_list, dtype=object),
    )
    print(f"Extracted {features.shape} features → {cache_path}")
    return {"features": features, "y": y, "ids": ids, "splits": split_arr, "paths": paths}


DEFAULT_MODEL = None
DEFAULT_KIND = None
FEATURE_PACK = None

if usable:
    DEFAULT_MODEL, DEFAULT_KIND = load_dinov2_vits14()
    FEATURE_PACK = extract_features(DEFAULT_MODEL, DEFAULT_KIND, usable, CACHE_NAME)
else:
    print("No usable images; cannot extract features.")


feature device: cuda
Downloading: "https://github.com/facebookresearch/dinov2/zipball/main" to /root/.cache/torch/hub/main.zip


/root/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/swiglu_ffn.py:51: UserWarning: xFormers is not available (SwiGLU)
  warnings.warn("xFormers is not available (SwiGLU)")
/root/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/attention.py:33: UserWarning: xFormers is not available (Attention)
  warnings.warn("xFormers is not available (Attention)")
/root/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/block.py:40: UserWarning: xFormers is not available (Block)
  warnings.warn("xFormers is not available (Block)")


Downloading: "https://dl.fbaipublicfiles.com/dinov2/dinov2_vits14/dinov2_vits14_pretrain.pth" to /root/.cache/torch/hub/checkpoints/dinov2_vits14_pretrain.pth


100%|██████████| 84.2M/84.2M [00:00<00:00, 184MB/s]


Loaded facebookresearch/dinov2 dinov2_vits14
Extracted (204, 384) features → /content/adiretrust-app/model/runs/dinov2_vits14_224.npz


## 6. Default: train a logistic regression probe

Regularized linear probe on frozen embeddings. This is the plan’s first real score when both classes are present. Fit on **train** only.


In [6]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import LabelEncoder
import joblib
import numpy as np

CLASS_ORDER = ["factory_print", "genuine_hand_dyed"]
probe = None
label_encoder = None
feat_dim = None
train_X = train_y = val_X = val_y = test_X = test_y = None
val_ids = test_ids = train_ids = None
val_paths = test_paths = None
proba_val = proba_test = None
classes_ = None


def split_xy(pack: dict, split: str):
    mask = pack["splits"] == split
    return pack["features"][mask], pack["y"][mask], pack["ids"][mask], pack["paths"][mask]


if FEATURE_PACK is None:
    print("Skipped fitting: no features.")
else:
    train_X, train_y, train_ids, _train_paths = split_xy(FEATURE_PACK, "train")
    val_X, val_y, val_ids, val_paths = split_xy(FEATURE_PACK, "val")
    test_X, test_y, test_ids, test_paths = split_xy(FEATURE_PACK, "test")
    feat_dim = int(FEATURE_PACK["features"].shape[1])
    print("train", train_X.shape, dict(Counter(train_y.tolist())))
    print("val  ", val_X.shape, dict(Counter(val_y.tolist())))
    print("test ", test_X.shape, dict(Counter(test_y.tolist())))

    present = sorted(set(train_y.tolist()))
    if len(present) < 2:
        print(
            "Cannot fit a two-class logistic regression: train only has "
            f"{present}. Add the missing Osun print photos and rerun."
        )
    else:
        label_encoder = LabelEncoder()
        # Prefer a stable class order when both labels exist.
        if set(present) == set(CLASS_ORDER):
            label_encoder.classes_ = np.array(CLASS_ORDER, dtype=object)
            train_enc = np.array([CLASS_ORDER.index(y) for y in train_y])
        else:
            train_enc = label_encoder.fit_transform(train_y)
        classes_ = list(label_encoder.classes_) if hasattr(label_encoder, "classes_") and label_encoder.classes_ is not None and len(getattr(label_encoder, "classes_", [])) else list(present)
        if set(present) == set(CLASS_ORDER):
            classes_ = CLASS_ORDER
            label_encoder.classes_ = np.array(CLASS_ORDER, dtype=object)

        probe = LogisticRegression(
            C=1.0,
            max_iter=2000,
            class_weight="balanced",
            solver="lbfgs",
        )
        probe.fit(train_X, [classes_.index(y) for y in train_y])
        print("Fitted LogisticRegression on", train_X.shape[0], "rows; classes =", classes_)

        def predict_proba(X):
            raw = probe.predict_proba(X)
            # Map columns to classes_
            col = {int(c): i for i, c in enumerate(probe.classes_)}
            ordered = np.zeros_like(raw)
            for i, name in enumerate(classes_):
                # probe.classes_ are integer encodings
                ordered[:, i] = raw[:, col[i]]
            return ordered

        if len(val_X):
            proba_val = predict_proba(val_X)
        if len(test_X):
            proba_test = predict_proba(test_X)
        print("Logistic regression is the default exported probe.")


train (151, 384) {'genuine_hand_dyed': 65, 'factory_print': 86}
val   (26, 384) {'genuine_hand_dyed': 12, 'factory_print': 14}
test  (27, 384) {'genuine_hand_dyed': 12, 'factory_print': 15}
Fitted LogisticRegression on 151 rows; classes = ['factory_print', 'genuine_hand_dyed']
Logistic regression is the default exported probe.


## 7. Expanded evaluation

Report precision and recall on **both** classes, the false-genuine count, a confusion matrix, coverage / abstain rate, and ROC-AUC **only when both classes are present** in that split. Numbers are computed from this run. Do not paste a batik-paper figure as an AdireTrust result.


In [7]:
from sklearn.metrics import confusion_matrix, roc_auc_score, precision_score, recall_score

GENUINE = "genuine_hand_dyed"
FACTORY = "factory_print"


def genuine_index(class_names: list[str]) -> int | None:
    if GENUINE in class_names:
        return class_names.index(GENUINE)
    return None


def decide(proba: np.ndarray, class_names: list[str], threshold: float):
    # Abstain when the winning class probability is below `threshold`.
    if proba is None or len(proba) == 0:
        return np.array([], dtype=object), np.array([], dtype=float)
    top = proba.max(axis=1)
    pred_idx = proba.argmax(axis=1)
    labels = np.array(
        [class_names[i] if p >= threshold else "uncertain" for i, p in zip(pred_idx, top)],
        dtype=object,
    )
    return labels, top


def per_class_pr(y_true, y_pred, label: str) -> dict:
    yt = np.array(y_true, dtype=object)
    yp = np.array(y_pred, dtype=object)
    support = int((yt == label).sum())
    pred_pos = int((yp == label).sum())
    tp = int(((yt == label) & (yp == label)).sum())
    precision = (tp / pred_pos) if pred_pos else None
    recall = (tp / support) if support else None
    return {
        "label": label,
        "support": support,
        "predicted": pred_pos,
        "tp": tp,
        "precision": precision,
        "recall": recall,
    }


def evaluate_split(name: str, y_true, proba, class_names, threshold: float) -> dict:
    result = {
        "split": name,
        "n": 0 if y_true is None else int(len(y_true)),
        "threshold": threshold,
        "class_counts": dict(Counter(y_true.tolist())) if y_true is not None and len(y_true) else {},
        "skipped": False,
        "reason": "",
    }
    if y_true is None or proba is None or len(y_true) == 0:
        result["skipped"] = True
        result["reason"] = "empty split or no probabilities"
        return result

    pred, conf = decide(proba, class_names, threshold)
    decided = pred != "uncertain"
    n = len(y_true)
    n_decided = int(decided.sum())
    coverage = n_decided / n if n else 0.0
    abstain_rate = 1.0 - coverage

    y_true_l = np.array(y_true, dtype=object)
    pred_d = pred[decided]
    true_d = y_true_l[decided]

    false_genuine_all = int(((pred == GENUINE) & (y_true_l == FACTORY)).sum())
    false_genuine_decided = int(((pred_d == GENUINE) & (true_d == FACTORY)).sum()) if n_decided else 0

    labels_for_cm = [c for c in [FACTORY, GENUINE] if c in set(y_true_l.tolist()) or c in set(pred.tolist())]
    cm = confusion_matrix(true_d, pred_d, labels=labels_for_cm) if n_decided else None

    pr_g = per_class_pr(true_d, pred_d, GENUINE) if n_decided else per_class_pr([], [], GENUINE)
    pr_p = per_class_pr(true_d, pred_d, FACTORY) if n_decided else per_class_pr([], [], FACTORY)

    auc = None
    auc_note = ""
    gidx = genuine_index(class_names)
    true_classes = set(y_true_l.tolist())
    if gidx is None:
        auc_note = "no genuine class in the probe"
    elif FACTORY not in true_classes or GENUINE not in true_classes:
        auc_note = "ROC-AUC skipped: this split does not contain both classes"
    else:
        try:
            auc = float(roc_auc_score((y_true_l == GENUINE).astype(int), proba[:, gidx]))
        except Exception as exc:  # noqa: BLE001
            auc_note = f"ROC-AUC skipped: {exc}"

    result.update(
        {
            "coverage": coverage,
            "abstain_rate": abstain_rate,
            "n_decided": n_decided,
            "n_uncertain": int((~decided).sum()),
            "false_genuine_all": false_genuine_all,
            "false_genuine_decided": false_genuine_decided,
            "precision_genuine": pr_g["precision"],
            "recall_genuine": pr_g["recall"],
            "support_genuine_decided": pr_g["support"],
            "precision_print": pr_p["precision"],
            "recall_print": pr_p["recall"],
            "support_print_decided": pr_p["support"],
            "confusion_matrix": cm.tolist() if cm is not None else None,
            "confusion_labels": labels_for_cm,
            "roc_auc": auc,
            "roc_auc_note": auc_note,
            "pred": pred,
            "confidence": conf,
        }
    )
    return result


def fmt(v, digits=3):
    if v is None:
        return "n/a"
    if isinstance(v, float):
        return f"{v:.{digits}f}"
    return str(v)


def print_eval(res: dict) -> None:
    print(f"\n=== {res['split']} @ threshold={res['threshold']} ===")
    if res.get("skipped"):
        print("  skipped:", res.get("reason"))
        return
    print("  n =", res["n"], "class counts:", res["class_counts"])
    print("  coverage =", fmt(res["coverage"]), "abstain_rate =", fmt(res["abstain_rate"]), "uncertain =", res["n_uncertain"])
    print("  precision/recall genuine:", fmt(res["precision_genuine"]), "/", fmt(res["recall_genuine"]))
    print("  precision/recall print:  ", fmt(res["precision_print"]), "/", fmt(res["recall_print"]))
    print("  false-genuine (all rows) =", res["false_genuine_all"], "; among decided =", res["false_genuine_decided"])
    print("  confusion", res["confusion_labels"], "=", res["confusion_matrix"])
    if res["roc_auc"] is not None:
        print("  ROC-AUC (genuine vs rest, no abstain) =", fmt(res["roc_auc"]))
    elif res["roc_auc_note"]:
        print(" ", res["roc_auc_note"])


eval_val_default = None
eval_test_default = None
if probe is None or classes_ is None:
    print("No fitted default probe; evaluation is skipped.")
else:
    eval_val_default = evaluate_split("val", val_y, proba_val, classes_, CONFIDENCE_THRESHOLD_DEFAULT)
    eval_test_default = evaluate_split("test", test_y, proba_test, classes_, CONFIDENCE_THRESHOLD_DEFAULT)
    print_eval(eval_val_default)
    print_eval(eval_test_default)



=== val @ threshold=0.6 ===
  n = 26 class counts: {'genuine_hand_dyed': 12, 'factory_print': 14}
  coverage = 0.962 abstain_rate = 0.038 uncertain = 1
  precision/recall genuine: 1.000 / 1.000
  precision/recall print:   1.000 / 1.000
  false-genuine (all rows) = 0 ; among decided = 0
  confusion ['factory_print', 'genuine_hand_dyed'] = [[13, 0], [0, 12]]
  ROC-AUC (genuine vs rest, no abstain) = 1.000

=== test @ threshold=0.6 ===
  n = 27 class counts: {'genuine_hand_dyed': 12, 'factory_print': 15}
  coverage = 1.000 abstain_rate = 0.000 uncertain = 0
  precision/recall genuine: 1.000 / 1.000
  precision/recall print:   1.000 / 1.000
  false-genuine (all rows) = 0 ; among decided = 0
  confusion ['factory_print', 'genuine_hand_dyed'] = [[15, 0], [0, 12]]
  ROC-AUC (genuine vs rest, no abstain) = 1.000


## 8. Sweep the abstain threshold on val, then apply it to test

Tune on **val** only. The app default **0.6** is still reported as a starting point. A simple rule prefers zero (then fewer) false-genuine calls, then higher genuine precision, then higher coverage. The chosen threshold is applied to **test** without retuning.


In [8]:
chosen_threshold = CONFIDENCE_THRESHOLD_DEFAULT
sweep_rows = []
eval_val_chosen = None
eval_test_chosen = None

if probe is None or classes_ is None or proba_val is None or len(val_y) == 0:
    print("Threshold sweep skipped (no fitted probe or empty val).")
    print(f"Keeping the app default threshold {CONFIDENCE_THRESHOLD_DEFAULT}.")
else:
    for t in THRESHOLD_GRID:
        row = evaluate_split("val", val_y, proba_val, classes_, t)
        sweep_rows.append(row)
        print(
            f"val t={t:.2f}  coverage={fmt(row['coverage'])}  "
            f"P_gen={fmt(row['precision_genuine'])}  R_gen={fmt(row['recall_genuine'])}  "
            f"P_print={fmt(row['precision_print'])}  R_print={fmt(row['recall_print'])}  "
            f"false_genuine={row['false_genuine_all']}"
        )

    def sort_key(row):
        # Prefer fewer false genuines, then higher genuine precision, then coverage.
        p = row["precision_genuine"]
        p_key = p if p is not None else -1.0
        return (row["false_genuine_all"], -p_key, -row["coverage"], row["threshold"])

    ranked = sorted(sweep_rows, key=sort_key)
    chosen_threshold = float(ranked[0]["threshold"])
    print(
        f"\nChosen val threshold: {chosen_threshold} "
        f"(rule: minimize false-genuine, then maximize genuine precision, then coverage)."
    )
    print(f"App default 0.6 is still reported alongside the chosen value.")

    eval_val_chosen = evaluate_split("val", val_y, proba_val, classes_, chosen_threshold)
    eval_test_chosen = evaluate_split("test", test_y, proba_test, classes_, chosen_threshold)
    print_eval(eval_val_chosen)
    print("\nTest at the val-chosen threshold (not retuned):")
    print_eval(eval_test_chosen)
    print("\nTest at the app default 0.6 (starting point, not retuned):")
    print_eval(eval_test_default)


val t=0.50  coverage=1.000  P_gen=1.000  R_gen=1.000  P_print=1.000  R_print=1.000  false_genuine=0
val t=0.55  coverage=0.962  P_gen=1.000  R_gen=1.000  P_print=1.000  R_print=1.000  false_genuine=0
val t=0.60  coverage=0.962  P_gen=1.000  R_gen=1.000  P_print=1.000  R_print=1.000  false_genuine=0
val t=0.65  coverage=0.962  P_gen=1.000  R_gen=1.000  P_print=1.000  R_print=1.000  false_genuine=0
val t=0.70  coverage=0.962  P_gen=1.000  R_gen=1.000  P_print=1.000  R_print=1.000  false_genuine=0
val t=0.75  coverage=0.962  P_gen=1.000  R_gen=1.000  P_print=1.000  R_print=1.000  false_genuine=0
val t=0.80  coverage=0.962  P_gen=1.000  R_gen=1.000  P_print=1.000  R_print=1.000  false_genuine=0

Chosen val threshold: 0.5 (rule: minimize false-genuine, then maximize genuine precision, then coverage).
App default 0.6 is still reported alongside the chosen value.

=== val @ threshold=0.5 ===
  n = 26 class counts: {'genuine_hand_dyed': 12, 'factory_print': 14}
  coverage = 1.000 abstain_rate 

## 9. Save the default probe

Small files only (`.joblib` + `label_map.json`). Feature caches stay under `model/runs/` and are gitignored.


In [9]:
from datetime import datetime, timezone
import json
import joblib

PROBE_PATH = WEIGHTS_DIR / "dinov2_vits14_logreg.joblib"
LABEL_MAP_PATH = WEIGHTS_DIR / "label_map.json"
saved_probe = False

if probe is None or classes_ is None or feat_dim is None:
    print("No probe to save. Add print photos so both classes exist, then rerun from the flags cell.")
else:
    joblib.dump(
        {
            "probe": probe,
            "classes": classes_,
            "backbone": "dinov2_vits14",
            "image_size": IMAGE_SIZE,
            "mean": IMAGENET_MEAN,
            "std": IMAGENET_STD,
            "feature_dim": feat_dim,
            "cache_name": CACHE_NAME,
        },
        PROBE_PATH,
    )
    label_map = {
        "classes": classes_,
        "backbone": "dinov2_vits14",
        "classifier": "logistic_regression",
        "image_size": IMAGE_SIZE,
        "mean": list(IMAGENET_MEAN),
        "std": list(IMAGENET_STD),
        "confidence_threshold_default": CONFIDENCE_THRESHOLD_DEFAULT,
        "confidence_threshold_chosen": chosen_threshold,
        "threshold_selection": "val sweep: min false-genuine, then max genuine precision, then coverage",
        "created_utc": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
        "n_train": int(len(train_y)) if train_y is not None else 0,
        "n_val": int(len(val_y)) if val_y is not None else 0,
        "n_test": int(len(test_y)) if test_y is not None else 0,
        "train_label_counts": dict(Counter(train_y.tolist())) if train_y is not None else {},
        "notes": "Frozen DINOv2 ViT-S/14 linear probe. Do not treat this as a certification model.",
    }
    LABEL_MAP_PATH.write_text(json.dumps(label_map, indent=2) + "\n")
    saved_probe = True
    print("Wrote", PROBE_PATH, "size", PROBE_PATH.stat().st_size, "bytes")
    print("Wrote", LABEL_MAP_PATH)


Wrote /content/adiretrust-app/model/weights/dinov2_vits14_logreg.joblib size 4181 bytes
Wrote /content/adiretrust-app/model/weights/label_map.json


## 10. Write `EVAL.md`

The report is generated from the numbers computed above. If a class is missing, the file says so. It is not a pitch-ready accuracy claim.


In [10]:
from datetime import datetime, timezone
from pathlib import Path

EVAL_PATH = REPO_ROOT / "model" / "EVAL.md"


def md_metric(v, digits=3) -> str:
    if v is None:
        return "n/a"
    if isinstance(v, float):
        return f"{v:.{digits}f}"
    return str(v)


def section_for(res: dict | None, heading: str) -> list[str]:
    lines = [f"### {heading}", ""]
    if res is None:
        lines += ["Not computed (no fitted probe or empty split).", ""]
        return lines
    if res.get("skipped"):
        lines += [f"Skipped: {res.get('reason')}", ""]
        return lines
    lines += [
        f"- n = {res['n']}; class counts = `{res['class_counts']}`",
        f"- threshold = {res['threshold']}",
        f"- coverage = {md_metric(res['coverage'])}; abstain rate = {md_metric(res['abstain_rate'])}; uncertain = {res['n_uncertain']}",
        f"- precision / recall on genuine: {md_metric(res['precision_genuine'])} / {md_metric(res['recall_genuine'])}",
        f"- precision / recall on factory print: {md_metric(res['precision_print'])} / {md_metric(res['recall_print'])}",
        f"- false-genuine count (all rows) = {res['false_genuine_all']}; among decided = {res['false_genuine_decided']}",
        f"- confusion matrix labels = `{res['confusion_labels']}`",
        f"- confusion matrix = `{res['confusion_matrix']}`",
    ]
    if res["roc_auc"] is not None:
        lines.append(f"- ROC-AUC (genuine vs rest, scored without abstain) = {md_metric(res['roc_auc'])}")
    else:
        lines.append(f"- ROC-AUC: {res.get('roc_auc_note') or 'n/a'}")
    lines.append("")
    return lines


lines = [
    "# AdireTrust probe evaluation",
    "",
    f"Generated (UTC): {datetime.now(timezone.utc).strftime('%Y-%m-%dT%H:%M:%SZ')}",
    "",
    "Default experiment: **frozen DINOv2 ViT-S/14 + logistic regression**.",
    "Metrics are computed from this run. This is not a certification result. Do not invent or overwrite these numbers.",
    "",
    "## Data",
    "",
    f"- Manifest labels: {len(labels)} rows; split rows: {len(splits)}",
    f"- Resolved image files: {len(resolved)}; missing files: {len(missing)}",
    f"- Missing by source: `{dict(Counter(str(r.get('source')) for r in missing))}`",
    f"- Usable (found + split, synthetics train-only): {len(usable)}",
    f"- Usable by split: `{dict(Counter(r['split'] for r in usable))}`",
    f"- Label counts by split: `{split_labels}`",
    "",
    "The public repo includes 81 Osun genuine photos and 24 train-only synthetics.",
    "The 100 Osun print photos listed in the manifest were not in the public source git history.",
    "Wikimedia Commons rows were unsplit and were not used.",
    "Current `piece_id` values are provisional (one image = one id).",
    "",
]

if not BOTH_CLASSES:
    lines += [
        "## Status",
        "",
        "A two-class probe was **not** fitted because training images do not cover both classes.",
        "Add the missing print photos (Drive mount cell) and rerun before quoting any accuracy.",
        "",
    ]

lines += [
    "## Thresholds",
    "",
    f"- App default (starting point): **{CONFIDENCE_THRESHOLD_DEFAULT}**",
    f"- Chosen on val (min false-genuine, then max genuine precision, then coverage): **{chosen_threshold}**",
    "",
    "The threshold is never selected on test. Test numbers below apply the val-chosen value and, separately, 0.6.",
    "",
    "## Val threshold sweep",
    "",
]
if sweep_rows:
    lines += [
        "| threshold | coverage | P genuine | R genuine | P print | R print | false-genuine |",
        "| ---: | ---: | ---: | ---: | ---: | ---: | ---: |",
    ]
    for row in sweep_rows:
        lines.append(
            f"| {row['threshold']:.2f} | {md_metric(row['coverage'])} | "
            f"{md_metric(row['precision_genuine'])} | {md_metric(row['recall_genuine'])} | "
            f"{md_metric(row['precision_print'])} | {md_metric(row['recall_print'])} | "
            f"{row['false_genuine_all']} |"
        )
    lines.append("")
else:
    lines += ["Sweep was not run.", ""]

lines += section_for(eval_val_default, f"Val at app default {CONFIDENCE_THRESHOLD_DEFAULT}")
lines += section_for(eval_test_default, f"Test at app default {CONFIDENCE_THRESHOLD_DEFAULT}")
if chosen_threshold != CONFIDENCE_THRESHOLD_DEFAULT or eval_val_chosen is not None:
    lines += section_for(eval_val_chosen, f"Val at chosen threshold {chosen_threshold}")
    lines += section_for(eval_test_chosen, f"Test at chosen threshold {chosen_threshold} (applied, not retuned)")

lines += [
    "## Export",
    "",
    f"- Probe saved: {saved_probe}",
    f"- `{PROBE_PATH.relative_to(REPO_ROOT) if saved_probe else 'not written'}`",
    f"- `{LABEL_MAP_PATH.relative_to(REPO_ROOT) if saved_probe else 'not written'}`",
    "",
    "## Optional flags used in this session",
    "",
    f"- RUN_KNN = {RUN_KNN}",
    f"- RUN_SIGLIP_EFFICIENTNET = {RUN_SIGLIP_EFFICIENTNET}",
    f"- RUN_DINOV2_VITB_PATCH = {RUN_DINOV2_VITB_PATCH}",
    f"- RUN_ZEROSHOT = {RUN_ZEROSHOT}",
    f"- RUN_STAGE2 = {RUN_STAGE2}",
    f"- RUN_FINETUNE = {RUN_FINETUNE}",
    f"- RUN_HEATMAP = {RUN_HEATMAP}",
    f"- RUN_MULTISHOT = {RUN_MULTISHOT}",
    "",
    "Optional-cell metrics are printed in the notebook when a flag is on; re-run this cell afterwards if you want them copied here by hand.",
    "",
]

EVAL_PATH.write_text("\n".join(lines) + "\n")
print("Wrote", EVAL_PATH)
print(EVAL_PATH.read_text()[:2000])
print("\n... (truncated in this print; open the file for the full report)")


Wrote /content/adiretrust-app/model/EVAL.md
# AdireTrust probe evaluation

Generated (UTC): 2026-11-25T11:24:17Z

Default experiment: **frozen DINOv2 ViT-S/14 + logistic regression**.
Metrics are computed from this run. This is not a certification result. Do not invent or overwrite these numbers.

## Data

- Manifest labels: 221 rows; split rows: 205
- Resolved image files: 204; missing files: 1
- Missing by source: `{'osun_artisan_workshop': 1}`
- Usable (found + split, synthetics train-only): 204
- Usable by split: `{'train': 151, 'test': 27, 'val': 26}`
- Label counts by split: `{'train': {'genuine_hand_dyed': 65, 'factory_print': 86}, 'val': {'genuine_hand_dyed': 12, 'factory_print': 14}, 'test': {'genuine_hand_dyed': 12, 'factory_print': 15}}`

The public repo includes 81 Osun genuine photos and 24 train-only synthetics.
The 100 Osun print photos listed in the manifest were not in the public source git history.
Wikimedia Commons rows were unsplit and were not used.
Current `piece_

## 11. Download weights and commit them

Download the two small files (or copy them to Drive), then on a machine with git access:

```bash
git add model/weights/dinov2_vits14_logreg.joblib model/weights/label_map.json model/EVAL.md
git commit -m "Add DINOv2 ViT-S/14 logistic probe and EVAL.md from Colab."
git push origin main
```

Do not commit `model/runs/` feature caches (gitignored). Hand the weights to David to wire into `POST /check`. The API image should still not import torch at startup until inference is actually loaded lazily.


In [11]:
try:
    from google.colab import files  # type: ignore

    if saved_probe:
        files.download(str(PROBE_PATH))
        files.download(str(LABEL_MAP_PATH))
    if EVAL_PATH.is_file():
        files.download(str(EVAL_PATH))
except ImportError:
    print("Not in Colab; files are already at:")
    print(" ", PROBE_PATH if saved_probe else "(probe not saved)")
    print(" ", LABEL_MAP_PATH if saved_probe else "(label map not saved)")
    print(" ", EVAL_PATH)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

---

# Optional experiments (off by default)

Set the matching `RUN_*` flag in **§4** to `True`, rerun that cell, then run the section you want. Compare probes on **val**, not test. The exported default remains DINOv2 ViT-S/14 + logistic regression unless you deliberately change the save cell.


## Optional A — k-NN vs logistic regression

Plan experiment 2: compare a cosine k-NN on the same frozen DINOv2 ViT-S/14 embeddings to the default logistic regression. Choose *k* on **val**. This cell does not overwrite the saved logreg probe.


In [12]:
from sklearn.neighbors import KNeighborsClassifier

knn_results = []
knn_best = None

if not RUN_KNN:
    print("Skipped. Set RUN_KNN = True in the flags cell and rerun this section.")
elif FEATURE_PACK is None or train_X is None or len(train_X) == 0:
    print("Skipped: no features.")
elif len(set(train_y.tolist())) < 2:
    print("Skipped: k-NN also needs both classes in train.")
else:
    y_train_idx = np.array([classes_.index(y) if classes_ else 0 for y in train_y])
    ks = [k for k in (3, 5, 7) if k <= len(train_X)]
    if not ks:
        ks = [1]
    for k in ks:
        knn = KNeighborsClassifier(n_neighbors=k, metric="cosine", weights="distance")
        knn.fit(train_X, y_train_idx)
        proba = None
        if val_X is not None and len(val_X):
            raw = knn.predict_proba(val_X)
            # columns follow knn.classes_
            col = {int(c): i for i, c in enumerate(knn.classes_)}
            ordered = np.zeros((len(val_X), len(classes_)))
            for i in range(len(classes_)):
                if i in col:
                    ordered[:, i] = raw[:, col[i]]
            proba = ordered
        ev = evaluate_split(f"val kNN k={k}", val_y, proba, classes_, CONFIDENCE_THRESHOLD_DEFAULT)
        knn_results.append({"k": k, "model": knn, "eval": ev, "proba_val": proba})
        print_eval(ev)

    def knn_key(item):
        ev = item["eval"]
        p = ev.get("precision_genuine")
        p_key = p if p is not None else -1.0
        fg = ev.get("false_genuine_all", 10**9)
        cov = ev.get("coverage") or 0.0
        return (fg, -p_key, -cov)

    knn_best = sorted(knn_results, key=knn_key)[0]
    print(f"\nBest k-NN on val: k={knn_best['k']} (same selection rule as the threshold sweep).")
    if eval_val_default is not None:
        print(
            "Compare to default logreg val @ 0.6: "
            f"P_gen={fmt(eval_val_default['precision_genuine'])} "
            f"false_genuine={eval_val_default['false_genuine_all']} "
            f"coverage={fmt(eval_val_default['coverage'])}"
        )
        print("Do not pick a winner on test. The saved default probe is still logistic regression.")


Skipped. Set RUN_KNN = True in the flags cell and rerun this section.


## Optional B — SigLIP and frozen EfficientNet-B0 probes

Same protocol as the default: freeze the backbone, cache embeddings, fit logistic regression, score val at 0.6. Models are loaded with `timm` when the flag is on. If a checkpoint cannot be downloaded, the cell records the error and continues.


In [13]:
optional_backbone_evals = []


def timm_embed_model(model_name: str):
    import timm

    model = timm.create_model(model_name, pretrained=True, num_classes=0)
    cfg = model.pretrained_cfg if hasattr(model, "pretrained_cfg") else {}
    mean = tuple(cfg.get("mean", IMAGENET_MEAN))
    std = tuple(cfg.get("std", IMAGENET_STD))
    input_size = cfg.get("input_size", (3, IMAGE_SIZE, IMAGE_SIZE))
    size = int(input_size[-1]) if input_size else IMAGE_SIZE
    model.eval().to(DEVICE)
    for p in model.parameters():
        p.requires_grad = False
    return model, mean, std, size


@torch.no_grad()
def extract_timm(model, records, mean, std, size, cache_stem, batch_size=16):
    cache_path = RUNS_DIR / f"{cache_stem}.npz"
    key_list = [(r["image_id"], r["split"], r["resolved_path"]) for r in records]
    if cache_path.is_file():
        cached = np.load(cache_path, allow_pickle=True)
        cached_keys = [tuple(x) for x in cached["keys"].tolist()]
        if cached_keys == key_list:
            print(f"Loaded cached features {cache_path} shape={cached['features'].shape}")
            return {
                "features": cached["features"],
                "y": cached["y"],
                "ids": cached["ids"],
                "splits": cached["splits"],
                "paths": cached["paths"],
            }
    feats = []
    batch = []
    for rec in records:
        im = load_rgb_224(rec["resolved_path"], size=size)
        batch.append(pil_to_normalized_tensor(im, mean=mean, std=std))
        if len(batch) == batch_size:
            x = torch.stack(batch, dim=0).to(DEVICE)
            feats.append(model(x).detach().cpu().numpy())
            batch = []
    if batch:
        x = torch.stack(batch, dim=0).to(DEVICE)
        feats.append(model(x).detach().cpu().numpy())
    features = np.concatenate(feats, axis=0)
    pack = {
        "features": features,
        "y": np.array([r["label"] for r in records], dtype=object),
        "ids": np.array([r["image_id"] for r in records], dtype=object),
        "splits": np.array([r["split"] for r in records], dtype=object),
        "paths": np.array([r["resolved_path"] for r in records], dtype=object),
    }
    np.savez_compressed(cache_path, **pack, keys=np.array(key_list, dtype=object))
    print(f"Extracted {features.shape} → {cache_path}")
    return pack


def fit_and_eval_logreg(name: str, pack: dict) -> dict | None:
    tr_X, tr_y, _, _ = split_xy(pack, "train")
    va_X, va_y, _, _ = split_xy(pack, "val")
    te_X, te_y, _, _ = split_xy(pack, "test")
    present = sorted(set(tr_y.tolist()))
    print(f"{name}: train {tr_X.shape} {dict(Counter(tr_y.tolist()))}")
    if len(present) < 2:
        print(f"{name}: skip fit, train has {present}")
        return None
    cls = CLASS_ORDER if set(present) == set(CLASS_ORDER) else present
    clf = LogisticRegression(C=1.0, max_iter=2000, class_weight="balanced", solver="lbfgs")
    clf.fit(tr_X, [cls.index(y) for y in tr_y])
    raw = clf.predict_proba(va_X) if len(va_X) else None
    proba = None
    if raw is not None:
        col = {int(c): i for i, c in enumerate(clf.classes_)}
        proba = np.zeros((len(va_X), len(cls)))
        for i in range(len(cls)):
            if i in col:
                proba[:, i] = raw[:, col[i]]
    ev = evaluate_split(f"val {name}", va_y, proba, cls, CONFIDENCE_THRESHOLD_DEFAULT)
    print_eval(ev)
    return {"name": name, "eval": ev, "classes": cls, "clf": clf}


SIGLIP_CANDIDATES = [
    "vit_base_patch16_siglip_224",
    "vit_base_patch16_siglip_512",
    "timm/ViT-B-16-SigLIP",
]
EFFNET_CANDIDATES = [
    "tf_efficientnet_b0.ns_jft_in1k",
    "efficientnet_b0.ra_in1k",
    "efficientnet_b0",
]

if not RUN_SIGLIP_EFFICIENTNET:
    print("Skipped. Set RUN_SIGLIP_EFFICIENTNET = True in the flags cell and rerun this section.")
elif not usable:
    print("Skipped: no usable images.")
else:
    import timm

    def first_working(names):
        last_err = None
        for name in names:
            try:
                model, mean, std, size = timm_embed_model(name)
                print(f"Loaded {name} (size={size}, mean={mean})")
                return name, model, mean, std, size
            except Exception as exc:  # noqa: BLE001
                last_err = exc
                print(f"Could not load {name}: {exc}")
        raise RuntimeError(f"No candidate loaded; last error: {last_err}")

    for group, candidates, stem in [
        ("siglip", SIGLIP_CANDIDATES, "siglip_224"),
        ("efficientnet_b0", EFFNET_CANDIDATES, "efficientnet_b0_224"),
    ]:
        try:
            name, model, mean, std, size = first_working(candidates)
            pack = extract_timm(model, usable, mean, std, size, stem)
            ev = fit_and_eval_logreg(group, pack)
            if ev:
                optional_backbone_evals.append(ev)
            del model
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
        except Exception as exc:  # noqa: BLE001
            print(f"{group} failed: {exc}")
    if optional_backbone_evals and eval_val_default is not None:
        print("\nVal comparison vs default DINOv2 ViT-S/14 logreg @ 0.6 (do not pick on test):")
        print("  default P_gen=", fmt(eval_val_default["precision_genuine"]), "false_genuine=", eval_val_default["false_genuine_all"])
        for item in optional_backbone_evals:
            ev = item["eval"]
            print(" ", item["name"], "P_gen=", fmt(ev["precision_genuine"]), "false_genuine=", ev["false_genuine_all"])


Skipped. Set RUN_SIGLIP_EFFICIENTNET = True in the flags cell and rerun this section.


## Optional C — DINOv2 ViT-B and patch tokens

Try the larger DINOv2 ViT-B/14 global embedding, and a simple patch-token representation (mean of patch tokens) on ViT-S and, if it loads, ViT-B. Needs more GPU memory; the cell skips a backbone that OOM’s rather than crashing the default run.


In [14]:
patch_evals = []


@torch.no_grad()
def extract_cls_and_patchmean(model, kind: str, records, cache_stem, batch_size=8):
    cache_path = RUNS_DIR / f"{cache_stem}_cls_patchmean.npz"
    key_list = [(r["image_id"], r["split"], r["resolved_path"]) for r in records]
    if cache_path.is_file():
        cached = np.load(cache_path, allow_pickle=True)
        if [tuple(x) for x in cached["keys"].tolist()] == key_list:
            print(f"Loaded {cache_path}")
            return {k: cached[k] for k in ("cls", "patch_mean", "y", "ids", "splits", "paths")}

    cls_list, patch_list = [], []
    batch = []
    for rec in records:
        im = load_rgb_224(rec["resolved_path"])
        batch.append(pil_to_normalized_tensor(im))
        if len(batch) == batch_size:
            x = torch.stack(batch, dim=0).to(DEVICE)
            cls_b, patch_b = _forward_cls_patch(model, kind, x)
            cls_list.append(cls_b)
            patch_list.append(patch_b)
            batch = []
    if batch:
        x = torch.stack(batch, dim=0).to(DEVICE)
        cls_b, patch_b = _forward_cls_patch(model, kind, x)
        cls_list.append(cls_b)
        patch_list.append(patch_b)
    pack = {
        "cls": np.concatenate(cls_list, axis=0),
        "patch_mean": np.concatenate(patch_list, axis=0),
        "y": np.array([r["label"] for r in records], dtype=object),
        "ids": np.array([r["image_id"] for r in records], dtype=object),
        "splits": np.array([r["split"] for r in records], dtype=object),
        "paths": np.array([r["resolved_path"] for r in records], dtype=object),
    }
    np.savez_compressed(cache_path, **pack, keys=np.array(key_list, dtype=object))
    print(f"Extracted cls {pack['cls'].shape} patch_mean {pack['patch_mean'].shape} → {cache_path}")
    return pack


def _forward_cls_patch(model, kind: str, x: torch.Tensor):
    if kind == "hub" and hasattr(model, "forward_features"):
        out = model.forward_features(x)
    elif hasattr(model, "forward_features"):
        out = model.forward_features(x)
        if not isinstance(out, dict):
            # timm DINOv2 often returns a tensor already pooled; try prefix
            if hasattr(model, "forward_features") and hasattr(model, "norm"):
                # fall through to tokens if available
                pass
    else:
        out = model(x)

    if isinstance(out, dict):
        cls = out.get("x_norm_clstoken")
        patches = out.get("x_norm_patchtokens")
        if cls is None:
            raise RuntimeError(f"No cls token in feature dict keys={list(out.keys())}")
        if patches is None:
            raise RuntimeError("No patch tokens in feature dict")
        patch_mean = patches.mean(dim=1)
        return cls.detach().cpu().numpy(), patch_mean.detach().cpu().numpy()

    # last-resort: some timm models expose get_intermediate_layers
    if hasattr(model, "get_intermediate_layers"):
        tokens = model.get_intermediate_layers(x, n=1, reshape=False, return_class_token=True)
        # API varies; try a common layout
        raise RuntimeError("Model did not return a token dict; use the hub DINOv2 build for patch tokens.")
    raise RuntimeError("Could not read patch tokens from this model.")


def pack_from_arrays(base, feat_key: str) -> dict:
    return {
        "features": base[feat_key],
        "y": base["y"],
        "ids": base["ids"],
        "splits": base["splits"],
        "paths": base["paths"],
    }


if not RUN_DINOV2_VITB_PATCH:
    print("Skipped. Set RUN_DINOV2_VITB_PATCH = True in the flags cell and rerun this section.")
elif not usable:
    print("Skipped: no usable images.")
else:
    # ViT-S patch-mean (reload or reuse hub model)
    try:
        model_s, kind_s = (DEFAULT_MODEL, DEFAULT_KIND) if DEFAULT_MODEL is not None else load_dinov2_vits14()
        base_s = extract_cls_and_patchmean(model_s, kind_s, usable, "dinov2_vits14_224")
        ev = fit_and_eval_logreg("dinov2_vits14_patch_mean", pack_from_arrays(base_s, "patch_mean"))
        if ev:
            patch_evals.append(ev)
    except Exception as exc:  # noqa: BLE001
        print("ViT-S patch tokens failed:", exc)

    model_b = None
    try:
        try:
            model_b = torch.hub.load("facebookresearch/dinov2", "dinov2_vitb14", pretrained=True)
            kind_b = "hub"
            print("Loaded facebookresearch/dinov2 dinov2_vitb14")
        except Exception as exc:  # noqa: BLE001
            print("hub ViT-B failed, trying timm:", exc)
            import timm

            model_b = timm.create_model("vit_base_patch14_dinov2.lvd142m", pretrained=True, num_classes=0)
            kind_b = "timm"
            print("Loaded timm vit_base_patch14_dinov2.lvd142m")
        model_b.eval().to(DEVICE)
        for p in model_b.parameters():
            p.requires_grad = False
        base_b = extract_cls_and_patchmean(model_b, kind_b, usable, "dinov2_vitb14_224", batch_size=8)
        ev_cls = fit_and_eval_logreg("dinov2_vitb14_cls", pack_from_arrays(base_b, "cls"))
        ev_pt = fit_and_eval_logreg("dinov2_vitb14_patch_mean", pack_from_arrays(base_b, "patch_mean"))
        if ev_cls:
            patch_evals.append(ev_cls)
        if ev_pt:
            patch_evals.append(ev_pt)
    except torch.cuda.OutOfMemoryError:
        print("DINOv2 ViT-B ran out of GPU memory. Skipping ViT-B; the default ViT-S run is unchanged.")
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    except Exception as exc:  # noqa: BLE001
        print("DINOv2 ViT-B failed:", exc)
    finally:
        del model_b
        if torch.cuda.is_available():
            torch.cuda.empty_cache()


Skipped. Set RUN_DINOV2_VITB_PATCH = True in the flags cell and rerun this section.


## Optional D — CLIP / SigLIP zero-shot with the plan prompts

Plan experiment 1. Text prompts are taken verbatim from the engineering plan. This cell does not train. It only runs if `RUN_ZEROSHOT` is True, and it installs `open_clip_torch` at that point (not in the default path).


In [15]:
from PIL import Image
import numpy as np
import torch

ZERO_SHOT_PROMPTS = {
    "genuine_hand_dyed": "hand-dyed adire with irregular resist and dye bleed",
    "factory_print": "machine-printed imitation with identical repeats and a pale back",
}
# Prefer the factory / genuine order used elsewhere.
ZERO_SHOT_ORDER = ["factory_print", "genuine_hand_dyed"]
zero_shot_evals = []

if not RUN_ZEROSHOT:
    print("Skipped. Set RUN_ZEROSHOT = True in the flags cell and rerun this section.")
    print("Prompts that will be used:")
    for key in ZERO_SHOT_ORDER:
        print(f"  {key}: {ZERO_SHOT_PROMPTS[key]!r}")
elif not usable:
    print("Skipped: no usable images to score.")
else:
    import importlib
    import subprocess
    import sys

    try:
        import open_clip  # noqa: F401
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "open_clip_torch"])
        import open_clip

    candidates = [
        ("ViT-B-16-SigLIP", "webli"),
        ("ViT-B-16", "openai"),
        ("ViT-B-32", "openai"),
    ]
    model_z = preprocess_z = tokenizer_z = None
    loaded_name = None
    last_err = None
    for name, pretrained in candidates:
        try:
            model_z, _, preprocess_z = open_clip.create_model_and_transforms(name, pretrained=pretrained)
            tokenizer_z = open_clip.get_tokenizer(name)
            loaded_name = f"{name}:{pretrained}"
            print("Loaded open_clip", loaded_name)
            break
        except Exception as exc:  # noqa: BLE001
            last_err = exc
            print(f"Could not load {name} ({pretrained}): {exc}")
    if model_z is None:
        raise RuntimeError(f"No CLIP/SigLIP checkpoint loaded; last error: {last_err}")

    model_z = model_z.eval().to(DEVICE)
    for p in model_z.parameters():
        p.requires_grad = False
    text_tokens = tokenizer_z([ZERO_SHOT_PROMPTS[k] for k in ZERO_SHOT_ORDER]).to(DEVICE)
    with torch.no_grad():
        text_feat = model_z.encode_text(text_tokens)
        text_feat = text_feat / text_feat.norm(dim=-1, keepdim=True)

    def zero_shot_proba(records: list[dict]) -> tuple[np.ndarray, np.ndarray]:
        probs = []
        ys = []
        batch_im = []
        batch_y = []
        for rec in records:
            im = Image.open(rec["resolved_path"]).convert("RGB")
            batch_im.append(preprocess_z(im))
            batch_y.append(rec["label"])
            if len(batch_im) == 16:
                x = torch.stack(batch_im, dim=0).to(DEVICE)
                with torch.no_grad():
                    img_feat = model_z.encode_image(x)
                    img_feat = img_feat / img_feat.norm(dim=-1, keepdim=True)
                    logits = img_feat @ text_feat.T
                    probs.append(torch.softmax(logits, dim=-1).cpu().numpy())
                ys.extend(batch_y)
                batch_im, batch_y = [], []
        if batch_im:
            x = torch.stack(batch_im, dim=0).to(DEVICE)
            with torch.no_grad():
                img_feat = model_z.encode_image(x)
                img_feat = img_feat / img_feat.norm(dim=-1, keepdim=True)
                logits = img_feat @ text_feat.T
                probs.append(torch.softmax(logits, dim=-1).cpu().numpy())
            ys.extend(batch_y)
        return (np.concatenate(probs, axis=0) if probs else np.zeros((0, 2))), np.array(ys, dtype=object)

    print("Prompts:")
    for key in ZERO_SHOT_ORDER:
        print(f"  {key}: {ZERO_SHOT_PROMPTS[key]!r}")

    for split_name in ("val", "test"):
        recs = [r for r in usable if r["split"] == split_name]
        if not recs:
            print(f"{split_name}: no images")
            continue
        proba, y_true = zero_shot_proba(recs)
        ev = evaluate_split(
            f"{split_name} zero-shot {loaded_name}",
            y_true,
            proba,
            ZERO_SHOT_ORDER,
            CONFIDENCE_THRESHOLD_DEFAULT,
        )
        zero_shot_evals.append(ev)
        print_eval(ev)
    print("Zero-shot is an inspection / baseline only. It does not replace the fitted logreg probe.")
    del model_z
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


Skipped. Set RUN_ZEROSHOT = True in the flags cell and rerun this section.
Prompts that will be used:
  factory_print: 'machine-printed imitation with identical repeats and a pale back'
  genuine_hand_dyed: 'hand-dyed adire with irregular resist and dye bleed'


## Optional E — Stage-2 related-task data (DTD / batik / AITEX note)

Stage 2 of the plan is an intermediate related task (batik tulis/cap vs print, DTD, AITEX, optionally Wax-MNIST / TILDA). This cell **never auto-downloads** those datasets. It only inventories local folders if you already have them and `RUN_STAGE2` is True.


In [16]:
STAGE2_RELATIVE = [
    "data/raw/stage2/batik",
    "data/raw/stage2/batik_tulis",
    "data/raw/stage2/batik_cap",
    "data/raw/stage2/batik_print",
    "data/raw/stage2/dtd",
    "data/raw/stage2/aitex",
    "data/raw/stage2/wax_mnist",
    "data/raw/stage2/tilda",
]
STAGE2_DRIVE_DIR = ""  # optional extra root, e.g. "/content/drive/MyDrive/AdireTrust/stage2"
IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".webp", ".bmp", ".tif", ".tiff"}
stage2_inventory = {}


def _count_tree(root: Path) -> dict:
    files = [p for p in root.rglob("*") if p.is_file() and p.suffix.lower() in IMAGE_EXTS]
    by_top = Counter()
    for p in files:
        try:
            rel = p.relative_to(root)
            top = rel.parts[0] if len(rel.parts) > 1 else "_files"
        except ValueError:
            top = "_files"
        by_top[top] += 1
    return {"n_images": len(files), "by_subfolder": dict(by_top), "path": str(root)}


if not RUN_STAGE2:
    print("Skipped. Set RUN_STAGE2 = True in the flags cell to inventory local Stage-2 folders.")
    print("This cell will still not download batik, DTD, AITEX, Wax-MNIST, or TILDA.")
    print("Expected local layouts (any of these, if you already have the data):")
    for rel in STAGE2_RELATIVE:
        print(" ", REPO_ROOT / rel)
    if STAGE2_DRIVE_DIR:
        print(" Drive extra root:", STAGE2_DRIVE_DIR)
else:
    search_roots = [REPO_ROOT / rel for rel in STAGE2_RELATIVE]
    search_roots.append(REPO_ROOT / "data" / "raw" / "stage2")
    if STAGE2_DRIVE_DIR:
        search_roots.append(Path(STAGE2_DRIVE_DIR).expanduser())
    seen = set()
    for root in search_roots:
        key = str(root.resolve()) if root.exists() else str(root)
        if key in seen:
            continue
        seen.add(key)
        if root.is_dir():
            info = _count_tree(root)
            stage2_inventory[str(root)] = info
            print(f"FOUND {root}: {info['n_images']} images  subfolders={info['by_subfolder']}")
        else:
            print(f"not present: {root}")
    total = sum(v["n_images"] for v in stage2_inventory.values())
    print(f"\nStage-2 local images discovered: {total}")
    if total == 0:
        print(
            "No local Stage-2 data. To use this stage later, copy batik / DTD / AITEX "
            "(with consent and licenses) into data/raw/stage2/ yourself. "
            "Do not treat a batik-paper number as an AdireTrust result."
        )
    else:
        print(
            "Local Stage-2 files are present. A related-task pretrain is optional and not "
            "wired into the default field-photo probe in this notebook. Keep field val/test "
            "as the reported AdireTrust score."
        )


Skipped. Set RUN_STAGE2 = True in the flags cell to inventory local Stage-2 folders.
This cell will still not download batik, DTD, AITEX, Wax-MNIST, or TILDA.
Expected local layouts (any of these, if you already have the data):
  /content/adiretrust-app/data/raw/stage2/batik
  /content/adiretrust-app/data/raw/stage2/batik_tulis
  /content/adiretrust-app/data/raw/stage2/batik_cap
  /content/adiretrust-app/data/raw/stage2/batik_print
  /content/adiretrust-app/data/raw/stage2/dtd
  /content/adiretrust-app/data/raw/stage2/aitex
  /content/adiretrust-app/data/raw/stage2/wax_mnist
  /content/adiretrust-app/data/raw/stage2/tilda


## Optional F — Train-only augmentation + last-block fine-tune (OFF by default)

Plan experiment 3: fine-tune **only the val-winning backbone**, last blocks first, ~1e-5, early stopping on val precision-on-genuine. The plan says **do not full-network fine-tune on fewer than 300 pieces**. This cell stays **off** unless you set `RUN_FINETUNE = True`. It uses the default DINOv2 ViT-S/14, unfreezes only the last transformer block, and keeps the saved default probe as logistic regression unless you later export this run yourself.


In [17]:
FINETUNE_LR = 1e-5
FINETUNE_EPOCHS = 8
FINETUNE_PATIENCE = 3
FINETUNE_MIN_PIECES_PLAN = 300

if not RUN_FINETUNE:
    print(
        "OFF by default. The plan: fine-tune only the val-winning backbone, "
        "last blocks first at ~1e-5, early-stop on val precision-on-genuine. "
        f"Do not full-network fine-tune on fewer than {FINETUNE_MIN_PIECES_PLAN} pieces. "
        "The current field set is small and print photos may still be missing. "
        "Set RUN_FINETUNE = True in the flags cell only if you accept that risk "
        "and both classes are present in train and val."
    )
elif not usable:
    print("Fine-tune skipped: no usable images.")
elif not BOTH_CLASSES or len(set(r["label"] for r in usable if r["split"] == "val")) < 2:
    print("Fine-tune skipped: need both classes in train and val.")
else:
    n_pieces = len({r["piece_id"] for r in usable})
    print(
        f"WARNING: fine-tuning last block on {n_pieces} pieces "
        f"(plan threshold for full-network FT is {FINETUNE_MIN_PIECES_PLAN}). "
        "This run unfreezes only the last transformer block + a linear head. "
        "It does not overwrite model/weights/dinov2_vits14_logreg.joblib."
    )
    import copy
    import torch.nn as nn

    # Reload a fresh backbone so the frozen feature extractor for the default probe stays intact.
    ft_model, ft_kind = load_dinov2_vits14()
    for p in ft_model.parameters():
        p.requires_grad = False
    unfrozen = []
    if hasattr(ft_model, "blocks"):
        for p in ft_model.blocks[-1].parameters():
            p.requires_grad = True
            unfrozen.append(p)
    if not unfrozen:
        # fallback: unfreeze the last child of a blocks/layer module
        for name, mod in reversed(list(ft_model.named_children())):
            if any(True for _ in mod.parameters()) and name.lower() in {"blocks", "layer", "layers", "encoder"}:
                kids = list(mod.children())
                if kids:
                    for p in kids[-1].parameters():
                        p.requires_grad = True
                        unfrozen.append(p)
                break
    if hasattr(ft_model, "norm"):
        for p in ft_model.norm.parameters():
            p.requires_grad = True
            unfrozen.append(p)
    if not unfrozen:
        raise RuntimeError("Could not locate a last transformer block to unfreeze; aborting fine-tune.")
    print("Unfrozen parameter tensors:", len(unfrozen))

    # Probe feature dim from one frozen forward (still valid once last block is unfrozen).
    ft_train_ds = AdireClothDataset(usable, "train", augment=True, class_names=CLASS_ORDER)
    ft_val_ds = AdireClothDataset(usable, "val", augment=False, class_names=CLASS_ORDER)
    ft_train_loader = DataLoader(ft_train_ds, batch_size=8, shuffle=True, num_workers=0, collate_fn=adire_collate)
    ft_val_loader = DataLoader(ft_val_ds, batch_size=8, shuffle=False, num_workers=0, collate_fn=adire_collate)

    with torch.no_grad():
        x0, _, _ = ft_train_ds[0]
        feat0 = embed_batch_dinov2(ft_model, ft_kind, x0.unsqueeze(0).to(DEVICE))
        ft_dim = int(feat0.shape[-1])
    head = nn.Linear(ft_dim, len(CLASS_ORDER)).to(DEVICE)
    params = [p for p in ft_model.parameters() if p.requires_grad] + list(head.parameters())
    optim = torch.optim.AdamW(params, lr=FINETUNE_LR, weight_decay=0.01)
    loss_fn = nn.CrossEntropyLoss()

    def _forward_feats(x):
        if ft_kind == "hub":
            out = ft_model(x)
            if isinstance(out, dict):
                out = out.get("x_norm_clstoken", next(iter(out.values())))
            return out
        out = ft_model(x)
        if isinstance(out, (tuple, list)):
            out = out[0]
        return out

    def _val_precision_genuine() -> float | None:
        ft_model.eval()
        head.eval()
        ys = []
        probs = []
        with torch.no_grad():
            for xb, yb, _ in ft_val_loader:
                xb = xb.to(DEVICE)
                logits = head(_forward_feats(xb))
                proba = torch.softmax(logits, dim=-1).cpu().numpy()
                probs.append(proba)
                ys.extend([CLASS_ORDER[int(i)] for i in yb.tolist()])
        if not probs:
            return None
        ev = evaluate_split("val last-block-ft", np.array(ys, dtype=object), np.concatenate(probs, axis=0), CLASS_ORDER, CONFIDENCE_THRESHOLD_DEFAULT)
        print_eval(ev)
        return ev["precision_genuine"]

    best_p = None
    best_state = None
    bad = 0
    for epoch in range(1, FINETUNE_EPOCHS + 1):
        ft_model.train()
        head.train()
        running = 0.0
        n_seen = 0
        for xb, yb, _ in ft_train_loader:
            xb = xb.to(DEVICE)
            yb = yb.to(DEVICE)
            optim.zero_grad(set_to_none=True)
            logits = head(_forward_feats(xb))
            loss = loss_fn(logits, yb)
            loss.backward()
            optim.step()
            running += float(loss.item()) * len(yb)
            n_seen += len(yb)
        print(f"epoch {epoch}/{FINETUNE_EPOCHS} train_loss={running / max(n_seen, 1):.4f}")
        p_gen = _val_precision_genuine()
        score = p_gen if p_gen is not None else -1.0
        if best_p is None or score > best_p:
            best_p = score
            best_state = {
                "head": copy.deepcopy(head.state_dict()),
                "block": copy.deepcopy(ft_model.state_dict()),
            }
            bad = 0
            print(f"  new best val precision-on-genuine={best_p}")
        else:
            bad += 1
            print(f"  no improvement ({bad}/{FINETUNE_PATIENCE})")
            if bad >= FINETUNE_PATIENCE:
                print("Early stopping.")
                break
    if best_state is not None:
        out_ft = RUNS_DIR / "dinov2_vits14_lastblock_finetune.pt"
        torch.save({"head": best_state["head"], "backbone": best_state["block"], "classes": CLASS_ORDER, "lr": FINETUNE_LR}, out_ft)
        print("Saved fine-tune checkpoint (gitignored under model/runs/):", out_ft)
        print("Default exported probe remains the frozen logistic regression.")
    del ft_model, head
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


OFF by default. The plan: fine-tune only the val-winning backbone, last blocks first at ~1e-5, early-stop on val precision-on-genuine. Do not full-network fine-tune on fewer than 300 pieces. The current field set is small and print photos may still be missing. Set RUN_FINETUNE = True in the flags cell only if you accept that risk and both classes are present in train and val.


## Optional G — DINOv2 patch-similarity heatmap

Inspection aid only, not proof of how a cloth was made. When `RUN_HEATMAP` is True, this cell overlays cosine similarity between the DINOv2 class token and each patch token on one genuine photo and one print photo if both exist, otherwise on whatever usable images are available.


In [18]:
if not RUN_HEATMAP:
    print("Skipped. Set RUN_HEATMAP = True in the flags cell and rerun this section.")
elif not usable:
    print("Skipped: no usable images.")
else:
    import math
    import matplotlib.pyplot as plt

    model_h, kind_h = (DEFAULT_MODEL, DEFAULT_KIND) if DEFAULT_MODEL is not None else load_dinov2_vits14()

    def pick(label: str) -> dict | None:
        for split in ("val", "test", "train"):
            for rec in usable:
                if rec["label"] == label and rec["split"] == split:
                    return rec
        return None

    picks = [r for r in (pick("genuine_hand_dyed"), pick("factory_print")) if r is not None]
    if not picks:
        picks = usable[:2]

    def patch_similarity_map(rec: dict):
        im = load_rgb_resized(rec["resolved_path"], size=IMAGE_SIZE)
        x = pil_to_normalized_tensor(im).unsqueeze(0).to(DEVICE)
        with torch.no_grad():
            if hasattr(model_h, "forward_features"):
                out = model_h.forward_features(x)
            else:
                raise RuntimeError("This DINOv2 build does not expose forward_features for patch tokens.")
            if not isinstance(out, dict) or "x_norm_clstoken" not in out or "x_norm_patchtokens" not in out:
                raise RuntimeError(f"Unexpected features: {type(out)} keys={getattr(out, 'keys', lambda: None)()}")
            cls = out["x_norm_clstoken"]
            patches = out["x_norm_patchtokens"]
            cls_n = cls / cls.norm(dim=-1, keepdim=True)
            patch_n = patches / patches.norm(dim=-1, keepdim=True)
            sim = (patch_n * cls_n.unsqueeze(1)).sum(dim=-1)[0].cpu().numpy()
        n = sim.shape[0]
        side = int(math.sqrt(n))
        if side * side != n:
            raise RuntimeError(f"Patch count {n} is not a square; cannot reshape a grid.")
        grid = sim.reshape(side, side)
        return im, grid

    nplot = len(picks)
    fig, axes = plt.subplots(nplot, 2, figsize=(8, 4 * nplot))
    if nplot == 1:
        axes = np.array([axes])
    for ax_row, rec in zip(axes, picks):
        im, grid = patch_similarity_map(rec)
        ax_row[0].imshow(im)
        ax_row[0].set_title(f"{rec['label']}  {rec['piece_id']}  ({rec['split']})")
        ax_row[0].axis("off")
        ax_row[1].imshow(im)
        ax_row[1].imshow(grid, cmap="inferno", alpha=0.55, extent=(0, IMAGE_SIZE, IMAGE_SIZE, 0))
        ax_row[1].set_title("cls-to-patch cosine (inspection only)")
        ax_row[1].axis("off")
    fig.tight_layout()
    out_png = RUNS_DIR / "dinov2_patch_similarity_heatmap.png"
    fig.savefig(out_png, dpi=140)
    plt.show()
    print("Wrote", out_png, "(gitignored under model/runs/). This heatmap is not a certificate.")


Skipped. Set RUN_HEATMAP = True in the flags cell and rerun this section.


## Optional H — Multi-shot aggregation when `view_tag` is real

The plan scores views separately, then may average probabilities for front / back / macro of the same `piece_id`. This cell runs only when `RUN_MULTISHOT` is True **and** at least some `view_tag` values are not `unknown` / empty. On the current Osun IDs (provisional, usually one image per piece, tags unknown) it will skip rather than invent a grouping.


In [19]:
REAL_TAGS = {"front", "back", "macro", "full", "detail", "obverse", "reverse"}

if not RUN_MULTISHOT:
    print("Skipped. Set RUN_MULTISHOT = True in the flags cell and rerun this section.")
else:
    tags = [str(r.get("view_tag") or "").strip().lower() for r in usable]
    known = [t for t in tags if t and t not in {"unknown", "nan", "none"}]
    real = [t for t in tags if t in REAL_TAGS]
    print("view_tag counts on usable rows:", dict(Counter(tags)))
    if not real:
        if not known:
            print(
                "All view_tag values are unknown or empty. Current piece_ids are provisional "
                "(one image per id). Multi-shot aggregation is skipped so we do not invent views."
            )
        else:
            print("Non-unknown tags found, but they are not in the expected front/back/macro set:", dict(Counter(known)))
            print("Skipping aggregation rather than guessing a mapping.")
    elif probe is None or classes_ is None or FEATURE_PACK is None:
        print("Need the default fitted probe and features before aggregating. Run sections 5-6 first.")
    else:
        gidx = classes_.index("genuine_hand_dyed") if "genuine_hand_dyed" in classes_ else 0
        # image-level probabilities from the default probe
        rows = []
        for split_name, X, y, ids in [
            ("val", val_X, val_y, val_ids),
            ("test", test_X, test_y, test_ids),
        ]:
            if X is None or len(X) == 0:
                continue
            proba = predict_proba(X) if "predict_proba" in globals() else None
            if proba is None:
                raw = probe.predict_proba(X)
                col = {int(c): i for i, c in enumerate(probe.classes_)}
                proba = np.zeros((len(X), len(classes_)))
                for i in range(len(classes_)):
                    if i in col:
                        proba[:, i] = raw[:, col[i]]
            rec_by_id = {r["piece_id"]: r for r in usable if r["split"] == split_name}
            # usable may have unique piece_id; also allow lookup by image_id
            rec_by_image = {r["image_id"]: r for r in usable if r["split"] == split_name}
            for i, piece in enumerate(ids.tolist()):
                rec = rec_by_id.get(str(piece)) or rec_by_image.get(str(piece))
                tag = (rec or {}).get("view_tag") or "unknown"
                rows.append(
                    {
                        "split": split_name,
                        "piece_id": str(piece),
                        "label": y[i],
                        "view_tag": str(tag),
                        "proba": proba[i],
                    }
                )
        # Group by piece_id, mean probability, then evaluate at 0.6
        from collections import defaultdict

        grouped = defaultdict(list)
        for row in rows:
            grouped[(row["split"], row["piece_id"])].append(row)
        for split_name in ("val", "test"):
            items = [v for k, v in grouped.items() if k[0] == split_name]
            if not items:
                continue
            y_true = np.array([g[0]["label"] for g in items], dtype=object)
            proba = np.stack([np.mean([r["proba"] for r in g], axis=0) for g in items], axis=0)
            n_multi = sum(1 for g in items if len(g) > 1)
            print(f"{split_name}: {len(items)} pieces, {n_multi} with more than one view")
            ev_img = evaluate_split(
                f"{split_name} image-level",
                np.array([r["label"] for r in rows if r["split"] == split_name], dtype=object),
                np.stack([r["proba"] for r in rows if r["split"] == split_name], axis=0),
                classes_,
                CONFIDENCE_THRESHOLD_DEFAULT,
            )
            ev_piece = evaluate_split(
                f"{split_name} piece-mean proba",
                y_true,
                proba,
                classes_,
                CONFIDENCE_THRESHOLD_DEFAULT,
            )
            print_eval(ev_img)
            print_eval(ev_piece)


Skipped. Set RUN_MULTISHOT = True in the flags cell and rerun this section.
